## Atividade 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Solução manufaturada
def ue(x):
    return x*(1.0 - x) + 0.15*np.sin(2.0*np.pi*x) + 0.10*np.exp(x)

def f(x):
    # f = u_e,xx
    return -2.0 - 0.15*(2.0*np.pi)**2*np.sin(2.0*np.pi*x) + 0.10*np.exp(x)

u_left = float(ue(0.0))
u_right = float(ue(1.0))

print("Condições de contorno:")
print(f"u(0) = {u_left:.8f}")
print(f"u(1) = {u_right:.8f}")

# Funções de forma
def N(xp, xi, i):
    last_i = len(xi) - 1
    if (i == 0 and xp <= xi[1]):
        y = (xi[1] - xp) / (xi[1] - xi[0])
    elif (i == last_i and xp >= xi[last_i - 1]):
        y = (xp - xi[last_i - 1]) / (xi[last_i] - xi[last_i - 1])
    elif (xp >= xi[i] and xp <= xi[i + 1]):
        y = (xi[i + 1] - xp) / (xi[i + 1] - xi[i])
    elif (xp <= xi[i] and xp >= xi[i - 1]):
        y = (xp - xi[i - 1]) / (xi[i] - xi[i - 1])
    else:
        y = 0.0
    return y

def dNdx(xp, xi, i):
    last_i = len(xi) - 1
    if (i == 0 and xp <= xi[1]):
        y = (-1.0) / (xi[1] - xi[0])
    elif (i == last_i and xp >= xi[last_i - 1]):
        y = (1.0) / (xi[last_i] - xi[last_i - 1])
    elif (xp >= xi[i] and xp <= xi[i + 1]):
        y = (-1.0) / (xi[i + 1] - xi[i])
    elif (xp <= xi[i] and xp >= xi[i - 1]):
        y = (1.0) / (xi[i] - xi[i - 1])
    else:
        y = 0.0
    return y

# Solver FEM
def solve_fem(n_elem):
    xi = np.linspace(0.0, 1.0, n_elem + 1)
    n_nodes = len(xi)

    # nós internos = incógnitas livres
    free_nodes = list(range(1, n_nodes - 1))
    n_free = len(free_nodes)

    K = np.zeros((n_free, n_free))
    F = np.zeros(n_free)

    # pontos onde as funções mudam de expressão
    breakpoints = xi[1:-1]

    def innerprod_a(xp, xi, a, b):
        return dNdx(xp, xi, a) * dNdx(xp, xi, b)

    def innerprod_f(xp, xi, a):
        return N(xp, xi, a) * f(xp)

    for ia, a in enumerate(free_nodes):
        for ib, b in enumerate(free_nodes):
            K[ia, ib] = quad(
                innerprod_a, 0.0, 1.0,
                args=(xi, a, b),
                points=breakpoints,
                limit=200
            )[0]

        F[ia] = (
            - quad(
                innerprod_f, 0.0, 1.0,
                args=(xi, a),
                points=breakpoints,
                limit=200
            )[0]
            - u_left * quad(
                innerprod_a, 0.0, 1.0,
                args=(xi, a, 0),
                points=breakpoints,
                limit=200
            )[0]
            - u_right * quad(
                innerprod_a, 0.0, 1.0,
                args=(xi, a, n_nodes - 1),
                points=breakpoints,
                limit=200
            )[0]
        )

    print(f"\nn = {n_elem}")
    print("cond(K) =", np.linalg.cond(K))

    d_internal = np.linalg.solve(K, F)

    d = np.zeros(n_nodes)
    d[0] = u_left
    d[-1] = u_right
    d[1:-1] = d_internal

    return xi, d

# Avaliar solução FEM em qualquer ponto
def evaluate_fem(xp, xi, d):
    val = 0.0
    for j in range(len(xi)):
        val += N(xp, xi, j) * d[j]
    return val

# Solver FDM
def solve_fdm(n_elem):
    x = np.linspace(0.0, 1.0, n_elem + 1)
    h = x[1] - x[0]
    n_int = n_elem - 1

    A = np.zeros((n_int, n_int))
    b = np.zeros(n_int)

    for j in range(n_int):
        if j > 0:
            A[j, j - 1] = 1.0
        A[j, j] = -2.0
        if j < n_int - 1:
            A[j, j + 1] = 1.0

        b[j] = h**2 * f(x[j + 1])

    # incorporar as condições de contorno
    b[0] -= u_left
    b[-1] -= u_right

    u = np.zeros(n_elem + 1)
    u[0] = u_left
    u[-1] = u_right
    u[1:-1] = np.linalg.solve(A, b)

    return x, u

# Erro médio quadrático (RMSE)
def rmse_fem(xi, d, n_dense=2000):
    x_dense = np.linspace(0.0, 1.0, n_dense)
    u_num = np.array([evaluate_fem(xp, xi, d) for xp in x_dense])
    u_ex = ue(x_dense)
    return np.sqrt(np.mean((u_num - u_ex)**2))

def rmse_fdm(x, u, n_dense=2000):
    x_dense = np.linspace(0.0, 1.0, n_dense)
    u_num = np.interp(x_dense, x, u)
    u_ex = ue(x_dense)
    return np.sqrt(np.mean((u_num - u_ex)**2))

# Plot das funções N_i e dN_i/dx
n_plot_shape = 6
xi_shape = np.linspace(0.0, 1.0, n_plot_shape + 1)
x_plot = np.linspace(0.0, 1.0, 600)

plt.figure(figsize=(8, 4.5))
for j in range(len(xi_shape)):
    Nj = np.array([N(xp, xi_shape, j) for xp in x_plot])
    plt.plot(x_plot, Nj, linewidth=2)
plt.grid(True)
plt.xlabel("x")
plt.ylabel("N_i(x)")
plt.title("Funções de forma lineares por parte")
plt.show()

plt.figure(figsize=(8, 4.5))
for j in range(len(xi_shape)):
    dNj = np.array([dNdx(xp, xi_shape, j) for xp in x_plot])
    plt.plot(x_plot, dNj, linewidth=2)
plt.grid(True)
plt.xlabel("x")
plt.ylabel("dN_i/dx")
plt.title("Derivadas das funções de forma")
plt.show()

# Solução manufaturada e f(x)
fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axs[0].plot(x_plot, ue(x_plot), linewidth=2, label=r"$u_e(x)$")
axs[0].set_ylabel(r"$u_e(x)$")
axs[0].set_title("Solução manufaturada")
axs[0].grid(True)
axs[0].legend()

axs[1].plot(x_plot, f(x_plot), "--", linewidth=2, label=r"$f(x)=u''_e(x)$")
axs[1].set_xlabel("x")
axs[1].set_ylabel(r"$f(x)$")
axs[1].set_title("Termo fonte")
axs[1].grid(True)
axs[1].legend()

plt.tight_layout()
plt.show()

# Comparação para n = 6
n = 6

xi_fem, d_fem = solve_fem(n)
x_fdm, u_fdm = solve_fdm(n)

u_fem_plot = np.array([evaluate_fem(xp, xi_fem, d_fem) for xp in x_plot])

err_fem_6 = rmse_fem(xi_fem, d_fem)
err_fdm_6 = rmse_fdm(x_fdm, u_fdm)

print(f"\nComparação para n = {n}")
print(f"RMSE FEM = {err_fem_6:.8e}")
print(f"RMSE FDM = {err_fdm_6:.8e}")

plt.figure(figsize=(9, 5))
plt.plot(x_plot, ue(x_plot), "k", linewidth=2, label="Solução exata")
plt.plot(x_plot, u_fem_plot, "r--", linewidth=2, label="FEM")
plt.plot(x_fdm, u_fdm, "bo-", linewidth=1.5, label="FDM (nós)")
plt.plot(xi_fem, d_fem, "rs", label="Nós FEM")
plt.grid(True)
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("Comparação entre solução exata, FEM e FDM para n = 6")
plt.legend()
plt.show()

# Convergência para n = 4, 8, 16, 32
n_values = [4, 8, 16, 32]
errors_fem = []
errors_fdm = []

for n in n_values:
    xi_fem, d_fem = solve_fem(n)
    x_fdm, u_fdm = solve_fdm(n)

    errors_fem.append(rmse_fem(xi_fem, d_fem))
    errors_fdm.append(rmse_fdm(x_fdm, u_fdm))

errors_fem = np.array(errors_fem)
errors_fdm = np.array(errors_fdm)

print("\nTabela de erros:")
print("n     RMSE_FEM         RMSE_FDM")
for n, ef, ed in zip(n_values, errors_fem, errors_fdm):
    print(f"{n:<5d} {ef:.8e}   {ed:.8e}")

# taxas observadas
rates_fem = [np.nan]
rates_fdm = [np.nan]
for i in range(1, len(n_values)):
    rates_fem.append(np.log(errors_fem[i-1] / errors_fem[i]) / np.log(2))
    rates_fdm.append(np.log(errors_fdm[i-1] / errors_fdm[i]) / np.log(2))

print("\nTaxas observadas de convergência:")
print("n     rate_FEM    rate_FDM")
for n, rf, rd in zip(n_values, rates_fem, rates_fdm):
    if np.isnan(rf):
        print(f"{n:<5d} {'-':>8}    {'-':>8}")
    else:
        print(f"{n:<5d} {rf:8.4f}    {rd:8.4f}")

plt.figure(figsize=(8, 4.5))
plt.plot(n_values, errors_fem, "rs-", linewidth=2, label="FEM")
plt.plot(n_values, errors_fdm, "bo-", linewidth=2, label="FDM")

plt.grid(True)
plt.xlabel("n")
plt.ylabel("RMSE")
plt.title("Convergência do FEM e do FDM")
plt.legend()
plt.show()

## Atividade 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Parâmetros do problema
L = 1.0
Va = 10.0
Vb = 0.0

eps_left = 3.0
eps_right = 1.0

n_nodes = 15
if n_nodes % 2 == 0:
    raise ValueError("Use n_nodes ímpar para ter um nó na interface z=L/2.")

z = np.linspace(0.0, L, n_nodes)
h = z[1] - z[0]
n_elem = n_nodes - 1

print("Nó central =", z[n_nodes // 2])
print("Passo h =", h)

# Permissividade relativa
def eps_r(x):
    return eps_left if x < L/2 else eps_right

def eps_r_vec(x):
    x = np.asarray(x)
    return np.where(x < L/2, eps_left, eps_right)

# Solução analítica
def phi_exact(x):
    x = np.asarray(x)

    L1 = L/2
    L2 = L/2

    D = (Va - Vb) / (L1/eps_left + L2/eps_right)

    phi_1 = Va - D * x / eps_left
    phi_2 = Va - D * (L1/eps_left + (x - L1)/eps_right)

    return np.where(x <= L/2, phi_1, phi_2)

# Função de forma linear
def N(xp, xi, i):
    last_i = len(xi) - 1

    if i == 0 and xp <= xi[1]:
        return (xi[1] - xp) / (xi[1] - xi[0])

    if i == last_i and xp >= xi[last_i - 1]:
        return (xp - xi[last_i - 1]) / (xi[last_i] - xi[last_i - 1])

    if i < last_i and xp >= xi[i] and xp <= xi[i + 1]:
        return (xi[i + 1] - xp) / (xi[i + 1] - xi[i])

    if i > 0 and xp <= xi[i] and xp >= xi[i - 1]:
        return (xp - xi[i - 1]) / (xi[i] - xi[i - 1])

    return 0.0

# Derivada da função de forma
def dNdx(xp, xi, i):
    last_i = len(xi) - 1

    if i == 0 and xp <= xi[1]:
        return -1.0 / (xi[1] - xi[0])

    if i == last_i and xp >= xi[last_i - 1]:
        return 1.0 / (xi[last_i] - xi[last_i - 1])

    if i < last_i and xp >= xi[i] and xp <= xi[i + 1]:
        return -1.0 / (xi[i + 1] - xi[i])

    if i > 0 and xp <= xi[i] and xp >= xi[i - 1]:
        return 1.0 / (xi[i] - xi[i - 1])

    return 0.0

# FEM
def integrando_fem(xp, xi, a, b):
    return eps_r(xp) * dNdx(xp, xi, a) * dNdx(xp, xi, b)

xi = z.copy()
pontos_quebra = xi[1:-1]

K_fem = np.zeros((n_nodes, n_nodes))

for a in range(n_nodes):
    for b in range(n_nodes):
        K_fem[a, b] = quad(
            integrando_fem,
            0.0,
            L,
            args=(xi, a, b),
            points=pontos_quebra,
            limit=200
        )[0]

free = np.arange(1, n_nodes - 1)

K_red = K_fem[np.ix_(free, free)]
F_red = np.zeros(len(free))

F_red -= K_fem[np.ix_(free, [0])].flatten() * Va
F_red -= K_fem[np.ix_(free, [n_nodes - 1])].flatten() * Vb

phi_fem = np.zeros(n_nodes)
phi_fem[0] = Va
phi_fem[-1] = Vb
phi_fem[1:-1] = np.linalg.solve(K_red, F_red)

# FDM conservativo
eps_half = np.array([
    eps_r(0.5 * (z[i] + z[i + 1]))
    for i in range(n_elem)
])

n_internal = n_nodes - 2

A_cons = np.zeros((n_internal, n_internal))
b_cons = np.zeros(n_internal)

for i in range(1, n_nodes - 1):
    row = i - 1

    eps_m = eps_half[i - 1]
    eps_p = eps_half[i]

    A_cons[row, row] = (eps_m + eps_p) / h**2

    if row - 1 >= 0:
        A_cons[row, row - 1] = -eps_m / h**2
    else:
        b_cons[row] += eps_m * Va / h**2

    if row + 1 < n_internal:
        A_cons[row, row + 1] = -eps_p / h**2
    else:
        b_cons[row] += eps_p * Vb / h**2

phi_fdm_cons = np.zeros(n_nodes)
phi_fdm_cons[0] = Va
phi_fdm_cons[-1] = Vb
phi_fdm_cons[1:-1] = np.linalg.solve(A_cons, b_cons)

# FDM por operador expandido
epsilon_nodes = eps_r_vec(z)

A_exp = np.zeros((n_nodes, n_nodes))
b_exp = np.zeros(n_nodes)

A_exp[0, 0] = 1.0
A_exp[-1, -1] = 1.0
b_exp[0] = Va
b_exp[-1] = Vb

for i in range(1, n_nodes - 1):
    eps_i = epsilon_nodes[i]
    eps_dif = epsilon_nodes[i - 1] - epsilon_nodes[i + 1]

    A_exp[i, i - 1] = eps_i / h**2 + eps_dif / (4.0 * h**2)
    A_exp[i, i] = -2.0 * eps_i / h**2
    A_exp[i, i + 1] = eps_i / h**2 - eps_dif / (4.0 * h**2)

phi_fdm_exp = np.linalg.solve(A_exp, b_exp)

# Erros
phi_ex = phi_exact(z)

def erro_max(v_num, v_ref):
    return np.max(np.abs(v_num - v_ref))

def erro_rms(v_num, v_ref):
    return np.sqrt(np.mean((v_num - v_ref)**2))

print("\nErros máximos")
print("FEM vs exata              =", erro_max(phi_fem, phi_ex))
print("FDM conservativo vs exata =", erro_max(phi_fdm_cons, phi_ex))
print("FDM expandido vs exata    =", erro_max(phi_fdm_exp, phi_ex))
print("FEM vs FDM conservativo   =", erro_max(phi_fem, phi_fdm_cons))
print("FEM vs FDM expandido      =", erro_max(phi_fem, phi_fdm_exp))

print("\nErros RMS")
print("FEM vs exata              =", erro_rms(phi_fem, phi_ex))
print("FDM conservativo vs exata =", erro_rms(phi_fdm_cons, phi_ex))
print("FDM expandido vs exata    =", erro_rms(phi_fdm_exp, phi_ex))

# Gráfico das soluções
z_dense = np.linspace(0.0, L, 1000)

plt.figure(figsize=(9, 5.5))
plt.plot(z_dense, phi_exact(z_dense), 'k-', linewidth=2, label='Solução exata')
plt.plot(z, phi_fem, 'ro-', label='FEM')
plt.plot(z, phi_fdm_cons, 'bs--', label='FDM conservativo')
plt.plot(z, phi_fdm_exp, 'g^-.', label='FDM operador expandido')
plt.axvline(L/2, color='gray', linestyle=':', label='Interface z=L/2')
plt.xlabel('z')
plt.ylabel(r'$\phi(z)$')
plt.title('Capacitor com dois dielétricos')
plt.grid(True)
plt.legend()
plt.show()

# Gráfico dos erros absolutos
plt.figure(figsize=(9, 5))
plt.plot(z, np.abs(phi_fem - phi_ex), 'ro-', label='|FEM - exata|')
plt.plot(z, np.abs(phi_fdm_cons - phi_ex), 'bs--', label='|FDM conservativo - exata|')
plt.plot(z, np.abs(phi_fdm_exp - phi_ex), 'g^-.', label='|FDM expandido - exata|')
plt.axvline(L/2, color='gray', linestyle=':')
plt.xlabel('z')
plt.ylabel('Erro absoluto')
plt.title('Erro absoluto nos nós')
plt.grid(True)
plt.legend()
plt.show()

# Gráfico da permissividade
eps_plot = eps_r_vec(z_dense)

plt.figure(figsize=(8, 3.5))
plt.plot(z_dense, eps_plot, linewidth=2)
plt.axvline(L/2, color='gray', linestyle=':')
plt.xlabel('z')
plt.ylabel(r'$\varepsilon_r(z)$')
plt.title('Permissividade relativa')
plt.grid(True)
plt.show()